# Object_Detection Phase 5 — YOLOv11 파인튜닝

**목적**: AI Hub 음식이미지 데이터 전체(800종)로 yolo11s.pt 파인튜닝 → 한식/지방특산물 직접 탐지

**대상 데이터셋**: AI Hub '비전영역, 음식이미지 및 정보소개 텍스트 데이터' (dataSetSn=71564)
- 총 232,087장 / 800종 / 4대분류(A특수외식 / B일반외식배달 / C끼니대체 / D음료차류)
- 라벨: JSON 바운딩박스(2d_annotation) 포함

**결과물**: `best.pt` → 로컬 `Object_Detection/models/korean_food_v1_best.pt`

**로컬 연동**: `config/detection_config.yaml` 모델 경로 한 줄만 교체

---
### 실행 순서
1. 런타임 유형 → GPU(A100) 설정
2. AI Hub 데이터를 Drive `aihub_food/` 에 업로드
3. 셀을 순서대로 실행
4. Step 7 완료 후 `best.pt` 다운로드 → 로컬 `models/` 에 저장

## Step 0 — 환경 설정 및 Drive 마운트

In [ ]:
# GPU 확인
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
print('GPU 이름:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT   = '/content/drive/MyDrive'
DATASET_DIR  = f'{DRIVE_ROOT}/aihub_food'       # AI Hub 데이터 업로드 위치
FINETUNE_DIR = f'{DRIVE_ROOT}/finetune_dataset'  # 변환된 YOLO 데이터셋 저장
RUNS_DIR     = f'{DRIVE_ROOT}/runs'              # 학습 결과 저장

for d in [FINETUNE_DIR, RUNS_DIR,
          f'{FINETUNE_DIR}/train/images', f'{FINETUNE_DIR}/train/labels',
          f'{FINETUNE_DIR}/val/images',   f'{FINETUNE_DIR}/val/labels']:
    os.makedirs(d, exist_ok=True)

print('Drive 마운트 완료')
print(f'AI Hub 데이터 경로: {DATASET_DIR}')
print(f'YOLO 데이터셋 경로: {FINETUNE_DIR}')
print(f'학습 결과 경로:     {RUNS_DIR}')

In [ ]:
!pip install ultralytics -q
print('ultralytics 설치 완료')

## Step 1 — AI Hub 전체 클래스 수집

> AI Hub JSON에서 음식명(`food_type.fc`)을 전부 읽어 **클래스 목록을 자동으로 구성**한다.  
> 800종 전체를 학습 대상으로 사용.

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

def collect_all_classes(dataset_dir: str) -> list:
    """
    AI Hub JSON 전체를 스캔하여 존재하는 음식명 목록을 수집.
    → 800종 전체를 클래스로 등록
    """
    json_files = list(Path(dataset_dir).rglob('*.json'))
    print(f'JSON 파일 총계: {len(json_files):,}개')

    class_count = defaultdict(int)
    error_count = 0

    for jf in json_files:
        try:
            with open(jf, encoding='utf-8') as f:
                data = json.load(f)
            fc = data.get('data', {}).get('food_type', {}).get('fc', '').strip()
            if fc:
                class_count[fc] += 1
        except Exception:
            error_count += 1

    # 이미지 수 기준 내림차순 정렬 → class_id 부여
    sorted_classes = sorted(class_count.items(), key=lambda x: -x[1])
    class_names = [cls for cls, _ in sorted_classes]

    print(f'\n발견된 클래스 수: {len(class_names)}종')
    print(f'파싱 오류: {error_count}개')
    print(f'\n=== 상위 20개 클래스 (이미지 수 기준) ===')
    for cls, cnt in sorted_classes[:20]:
        print(f'  {cls}: {cnt}장')

    return class_names, dict(class_count)


if os.path.exists(DATASET_DIR):
    ALL_CLASSES, CLASS_COUNT = collect_all_classes(DATASET_DIR)
    CLASS_TO_ID = {cls: idx for idx, cls in enumerate(ALL_CLASSES)}
    print(f'\n총 클래스: {len(ALL_CLASSES)}종 → data.yaml nc={len(ALL_CLASSES)}')
else:
    print(f'⚠️  {DATASET_DIR} 없음')
    print('AI Hub 데이터를 Drive aihub_food/ 폴더에 업로드 후 재실행하세요')

## Step 2 — AI Hub JSON → YOLO 포맷 변환 (전체 변환)

> **스킵 없음** — 모든 이미지와 라벨을 YOLO 포맷으로 변환한다.  
> 변환 실패(bbox 오류, 이미지 없음)만 제외.

In [ ]:
import shutil
import random

# ─────────────────────────────────────────────────────
# AI Hub JSON 라벨 구조 (PDF 확인)
# {
#   "data": {
#     "image_info":    { "file_name": "xxx.jpg", "width": 1280, "height": 960 },
#     "2d_annotation": { "x": 100, "y": 200, "width": 300, "height": 250 },
#     "food_type":     { "fc": "비빔밥", "loc": "" }
#   }
# }
#
# YOLO 포맷: class_id cx cy w h  (0~1 정규화, 중심 좌표)
# ─────────────────────────────────────────────────────

def convert_one(json_path: str) -> tuple:
    """
    AI Hub JSON 1개 → (class_id, yolo_label, img_path)
    변환 불가 시 None 반환
    """
    try:
        with open(json_path, encoding='utf-8') as f:
            raw = json.load(f)
        data = raw.get('data', {})

        # 음식명
        fc = data.get('food_type', {}).get('fc', '').strip()
        if not fc or fc not in CLASS_TO_ID:
            return None

        # 이미지 정보
        img_info = data.get('image_info', {})
        img_w    = float(img_info.get('width', 0))
        img_h    = float(img_info.get('height', 0))
        fname    = img_info.get('file_name', '')
        if img_w <= 0 or img_h <= 0 or not fname:
            return None

        # bbox 변환: AI Hub (x, y, w, h, 절대좌표 좌측상단)
        #          → YOLO (cx, cy, w, h, 정규화 중심좌표)
        ann = data.get('2d_annotation', {})
        x   = float(ann.get('x', 0))
        y   = float(ann.get('y', 0))
        bw  = float(ann.get('width', 0))
        bh  = float(ann.get('height', 0))
        if bw <= 0 or bh <= 0:
            return None

        cx = (x + bw / 2) / img_w
        cy = (y + bh / 2) / img_h
        nw = bw / img_w
        nh = bh / img_h

        # 유효 범위 검증
        if not all(0.0 < v <= 1.0 for v in [cx, cy, nw, nh]):
            return None

        class_id   = CLASS_TO_ID[fc]
        yolo_label = f'{class_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}'

        # 이미지 파일 경로 탐색 (JSON과 같은 폴더 우선)
        img_path = Path(json_path).parent / fname
        if not img_path.exists():
            candidates = list(Path(json_path).parent.rglob(fname))
            if not candidates:
                return None
            img_path = candidates[0]

        return class_id, yolo_label, str(img_path), fc

    except Exception:
        return None


def build_full_dataset(
    dataset_dir: str,
    output_dir: str,
    val_ratio: float = 0.2,
):
    """
    AI Hub 전체 데이터 → YOLO 데이터셋 구성 (클래스 필터링 없음)
    train:val = 8:2 분리
    """
    json_files = list(Path(dataset_dir).rglob('*.json'))
    print(f'변환 대상 JSON: {len(json_files):,}개')

    # 클래스별 레코드 수집
    class_records = defaultdict(list)
    skip_count = 0

    for i, jf in enumerate(json_files):
        if i % 10000 == 0:
            print(f'  진행: {i:,} / {len(json_files):,}', end='\r')
        result = convert_one(str(jf))
        if result is None:
            skip_count += 1
            continue
        class_id, yolo_label, img_path, class_name = result
        class_records[class_name].append((yolo_label, img_path))

    print(f'\n\n변환 성공: {sum(len(v) for v in class_records.values()):,}장')
    print(f'변환 실패(bbox 오류/이미지 없음): {skip_count:,}개')

    # 클래스별 현황 출력
    print(f'\n=== 클래스별 이미지 수 (하위 10개 — 데이터 부족 경고) ===')
    sorted_records = sorted(class_records.items(), key=lambda x: len(x[1]))
    for cls, recs in sorted_records[:10]:
        status = '⚠️ ' if len(recs) < 50 else '  '
        print(f'{status} {cls}: {len(recs)}장')

    # train/val 분리 및 파일 복사
    train_count = val_count = 0
    for class_name, records in class_records.items():
        random.shuffle(records)
        split     = max(1, int(len(records) * (1 - val_ratio)))
        train_set = records[:split]
        val_set   = records[split:]

        for split_name, split_recs in [('train', train_set), ('val', val_set)]:
            for yolo_label, img_src in split_recs:
                img_src  = Path(img_src)
                dst_img  = Path(output_dir) / split_name / 'images' / img_src.name
                dst_lbl  = Path(output_dir) / split_name / 'labels' / (img_src.stem + '.txt')
                shutil.copy2(str(img_src), str(dst_img))
                dst_lbl.write_text(yolo_label + '\n', encoding='utf-8')
                if split_name == 'train':
                    train_count += 1
                else:
                    val_count += 1

    print(f'\n=== 데이터셋 구성 완료 ===')
    print(f'train: {train_count:,}장')
    print(f'val:   {val_count:,}장')
    print(f'합계:  {train_count + val_count:,}장')
    return train_count, val_count


if os.path.exists(DATASET_DIR):
    train_count, val_count = build_full_dataset(
        dataset_dir=DATASET_DIR,
        output_dir=FINETUNE_DIR,
        val_ratio=0.2,
    )
else:
    print(f'⚠️  {DATASET_DIR} 없음 — 데이터 업로드 후 재실행')

## Step 3 — data.yaml 생성

In [ ]:
import yaml

# 전체 클래스 목록으로 data.yaml 생성
data_yaml_content = {
    'train': f'{FINETUNE_DIR}/train/images',
    'val':   f'{FINETUNE_DIR}/val/images',
    'nc':    len(ALL_CLASSES),
    'names': ALL_CLASSES,
}

yaml_path = f'{FINETUNE_DIR}/data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml_content, f, allow_unicode=True, default_flow_style=False)

print(f'data.yaml 생성 완료')
print(f'  경로:     {yaml_path}')
print(f'  nc:       {len(ALL_CLASSES)}  (전체 클래스 수)')
print(f'  train:    {FINETUNE_DIR}/train/images')
print(f'  val:      {FINETUNE_DIR}/val/images')
print(f'\n상위 10개 클래스 확인:')
for i, cls in enumerate(ALL_CLASSES[:10]):
    print(f'  {i:3d}: {cls}')

## Step 4 — 파인튜닝 학습

> base 모델: `yolo11s.pt` (현재 프로젝트 `detector.py` 기본값과 동일)  
> 학습 파라미터: `detection_config.yaml` conf/iou 설정값 동일 적용

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')  # detector.py 기본값과 동일

print('=== 학습 시작 ===')
print(f'base 모델:  yolo11s.pt')
print(f'클래스 수:  {len(ALL_CLASSES)}종')
print(f'data.yaml:  {FINETUNE_DIR}/data.yaml')
print(f'결과 저장:  {RUNS_DIR}/korean_food_v1/')
print()

results = model.train(
    data=f'{FINETUNE_DIR}/data.yaml',

    # ── 학습 설정 ──
    epochs=100,       # 전체 반복 횟수
    imgsz=640,        # 입력 이미지 크기
    batch=16,         # A100 기준 최적값 (메모리 부족 시 8로 줄일 것)
    device=0,         # GPU
    workers=4,        # 데이터 로딩 병렬 수

    # ── 조기 종료 ──
    patience=20,      # 20 epoch 개선 없으면 자동 종료

    # ── 저장 경로 ──
    project=RUNS_DIR,
    name='korean_food_v1',
    exist_ok=True,
    save=True,
    save_period=10,   # 10 epoch마다 체크포인트 저장 (세션 끊김 대비)

    # ── detection_config.yaml 설정값과 동일하게 맞춤 ──
    # inference.confidence_threshold: 0.5
    # inference.iou_threshold:        0.45
    # inference.max_det:              100
    conf=0.001,       # 학습 시엔 낮게 설정 (val 시 threshold 적용)
    iou=0.45,
    max_det=100,

    verbose=True,
)

print(f'\n=== 학습 완료 ===')
print(f'best.pt: {RUNS_DIR}/korean_food_v1/weights/best.pt')

## Step 5 — 검증 및 성능 확인

In [ ]:
from ultralytics import YOLO
import pandas as pd

best_pt = f'{RUNS_DIR}/korean_food_v1/weights/best.pt'
model_ft = YOLO(best_pt)

metrics = model_ft.val(
    data=f'{FINETUNE_DIR}/data.yaml',
    device=0,
    conf=0.5,    # detection_config.yaml confidence_threshold
    iou=0.45,    # detection_config.yaml iou_threshold
)

print('\n=== 전체 검증 결과 ===')
print(f'mAP@0.5:      {metrics.box.map50:.4f}  (목표: ≥ 0.60)')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'Precision:    {metrics.box.mp:.4f}')
print(f'Recall:       {metrics.box.mr:.4f}')

# 우리 프로젝트 주요 클래스 AP 별도 출력
KEY_CLASSES = [
    '굴비', '조기', '대게', '전복', '장어', '한우', '흑돼지',  # 지방특산물
    '비빔밥', '김치찌개', '된장찌개', '삼겹살', '불고기',       # 한식
    '사과', '딸기', '수박', '감귤',                            # 과일채소
]

print('\n=== 주요 클래스 AP@0.5 (stt_keywords / clip_queries 기준) ===')
if hasattr(metrics.box, 'ap_class_index'):
    rows = []
    for i, class_idx in enumerate(metrics.box.ap_class_index):
        cls_name = ALL_CLASSES[int(class_idx)] if int(class_idx) < len(ALL_CLASSES) else str(class_idx)
        if cls_name not in KEY_CLASSES:
            continue
        ap50 = float(metrics.box.ap50[i]) if hasattr(metrics.box, 'ap50') else 0.0
        rows.append({'클래스': cls_name, 'AP@0.5': round(ap50, 4)})
    if rows:
        df = pd.DataFrame(rows).sort_values('AP@0.5', ascending=False)
        print(df.to_string(index=False))
    else:
        print('주요 클래스 데이터 없음 — AI Hub 데이터에 해당 음식명 확인 필요')

# 합격 판정
if metrics.box.map50 >= 0.60:
    print(f'\n✅ mAP@0.5 = {metrics.box.map50:.4f} → 합격')
else:
    print(f'\n⚠️  mAP@0.5 = {metrics.box.map50:.4f} → 미달 — 데이터 추가 또는 epochs 증가 권장')

## Step 6 — A/B 비교 (기존 yolo11s.pt vs 파인튜닝 best.pt)

In [ ]:
from ultralytics import YOLO
import glob
from collections import Counter

# val 이미지 샘플 20장으로 A/B 비교
val_images = glob.glob(f'{FINETUNE_DIR}/val/images/*.jpg')[:20]

if not val_images:
    print('val 이미지 없음 — Step 2 완료 후 실행')
else:
    model_base = YOLO('yolo11s.pt')
    model_ft   = YOLO(f'{RUNS_DIR}/korean_food_v1/weights/best.pt')

    base_labels = Counter()
    ft_labels   = Counter()

    for img_path in val_images:
        for pred in model_base(img_path, conf=0.5, verbose=False):
            for box in pred.boxes:
                base_labels[pred.names[int(box.cls)]] += 1

        for pred in model_ft(img_path, conf=0.5, verbose=False):
            for box in pred.boxes:
                ft_labels[pred.names[int(box.cls)]] += 1

    print(f'=== A/B 비교 (val 샘플 {len(val_images)}장) ===')
    print(f'\n기존 yolo11s.pt 탐지 상위 10개:')
    for label, cnt in base_labels.most_common(10):
        print(f'  {label}: {cnt}건')

    print(f'\n파인튜닝 best.pt 탐지 상위 10개:')
    for label, cnt in ft_labels.most_common(10):
        print(f'  {label}: {cnt}건')

    # 한식/지방특산물 탐지 비교
    KEY_CLASSES = set([
        '굴비', '조기', '대게', '전복', '장어', '한우', '흑돼지',
        '비빔밥', '김치찌개', '된장찌개', '삼겹살', '불고기',
        '사과', '딸기', '수박', '감귤',
    ])
    base_key = sum(cnt for lbl, cnt in base_labels.items() if lbl in KEY_CLASSES)
    ft_key   = sum(cnt for lbl, cnt in ft_labels.items()   if lbl in KEY_CLASSES)

    print(f'\n=== 주요 클래스(한식/지방특산물) 탐지 건수 ===')
    print(f'기존 yolo11s.pt: {base_key}건')
    print(f'파인튜닝 best.pt: {ft_key}건')
    print(f'개선: +{ft_key - base_key}건')

    if ft_key > base_key:
        print('\n✅ 파인튜닝 효과 확인')
    else:
        print('\n⚠️  효과 미미 — AI Hub 음식명 매칭 및 데이터 품질 확인 필요')

## Step 7 — best.pt 다운로드 및 로컬 적용 안내

In [ ]:
import os

best_pt_path = f'{RUNS_DIR}/korean_food_v1/weights/best.pt'

if os.path.exists(best_pt_path):
    size_mb = os.path.getsize(best_pt_path) / 1024 / 1024
    print(f'✅ best.pt 확인')
    print(f'   경로: {best_pt_path}')
    print(f'   크기: {size_mb:.1f} MB')
    print()
    print('=' * 50)
    print('로컬 적용 방법')
    print('=' * 50)
    print()
    print('1. Drive에서 best.pt 다운로드')
    print('   내 드라이브/runs/korean_food_v1/weights/best.pt')
    print()
    print('2. 로컬 프로젝트에 저장')
    print('   Object_Detection/models/korean_food_v1_best.pt')
    print()
    print('3. config/detection_config.yaml 수정 (한 줄만)')
    print('   변경 전: name: yolo11s.pt')
    print('   변경 후: name: models/korean_food_v1_best.pt')
    print()
    print('4. 로컬 배치 실행')
    print('   python scripts/batch_detect.py \\')
    print('       --input-dir data/trailers_아름 \\')
    print('       --limit 10 --random')
    print()
    print('5. 기존 테스트 통과 확인')
    print('   python -m pytest tests/test_phase1_setup.py -v')
    print('   → 13/13 PASS 유지 필수')
    print()
    print('⚠️  models/*.pt 는 .gitignore 대상 — 커밋 금지')
    print('   팀원 공유: Drive 링크 또는 직접 전달')
else:
    print(f'⚠️  {best_pt_path} 없음 — Step 4 완료 후 재실행')

---
## 현재 파이프라인 연동 포인트

```
Object_Detection/
├── src/detector.py              ← Detector(model_name=...) 모델 경로만 변경
├── config/detection_config.yaml ← name: models/korean_food_v1_best.pt
├── models/
│   └── korean_food_v1_best.pt   ← .gitignore 대상
└── data/
    └── vod_detected_object.parquet  ← 800종 한식 라벨 포함 재생성
```

### parquet 스키마 변경 없음

| 컬럼 | 타입 | 변화 |
|------|------|------|
| `vod_id` | str | 동일 |
| `frame_ts` | float | 동일 |
| `label` | str | **800종 한식 라벨 추가** (비빔밥, 굴비, 대게 등) |
| `confidence` | float | 동일 |
| `bbox` | list[float] | 동일 |

Shopping_Ad 연동 코드 변경 없이 `label` 다양성만 증가.